# **ST 554 Spring 2026 Final Project**
## *Created by Cody Ashby on April 27, 2026*
### Fitting The Model

In [1]:
import pandas as pd
import numpy as np
import time
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, SQLTransformer, VectorAssembler, Binarizer, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder # set parallelism to 128 when doing CV!
from pyspark.ml.evaluation import RegressionEvaluator

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 22:37:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/27 22:37:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/27 22:37:21 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/27 22:37:21 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/27 22:37:21 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/27 22:37:21 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


In [2]:
power_data = pd.read_csv("power_ml_data.csv")

In [3]:
spark_power_data = spark.read.csv("power_ml_data.csv",inferSchema=True,header=True)
spark_power_data.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', IntegerType(), True), StructField('Hour', IntegerType(), True)])

In [4]:
sqlTrans=SQLTransformer(statement="SELECT Month, Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows, \
                                        Power_Zone_1, Power_Zone_2, Power_Zone_3 as label, CAST(Hour AS DOUBLE) AS Revised_Hour FROM __THIS__")

In [5]:
binary_Hour=Binarizer(threshold=6.5,inputCol="Revised_Hour",outputCol="Binary_Hour")

In [6]:
#Month_Index=StringIndexer(inputCol="Month",outputCol="Indexed_Month")
Month_OHE=OneHotEncoder(inputCol="Month",outputCol="Encoded_Month")

In [7]:
Weather_Vars=VectorAssembler(inputCols=['Temperature','Humidity','Wind_Speed','General_Diffuse_Flows','Diffuse_Flows'],outputCol='weather_features',handleInvalid='keep')
Weather_Comps=Weather_Vars.transform(spark_power_data)
PC_features=PCA(k=2,inputCol='weather_features',outputCol="Prin_Comps")
Fitted_PC=PC_features.fit(Weather_Comps)

In [8]:
total_features=VectorAssembler(inputCols=['Prin_Comps','Binary_Hour','Power_Zone_1','Power_Zone_2','Encoded_Month'],outputCol='features',handleInvalid='keep')

In [9]:
MLR=LinearRegression()
grand_pipeline=Pipeline(stages=[sqlTrans,binary_Hour,Month_OHE,Weather_Vars,PC_features,total_features,MLR])

In [10]:
EN_Param_Grid=ParamGridBuilder().addGrid(MLR.regParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]) \
                            .addGrid(MLR.elasticNetParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]).build()

In [11]:
ElasticNet_CrossVal=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=EN_Param_Grid,
                                   evaluator=RegressionEvaluator(labelCol='label',metricName='rmse'),
                                   numFolds=5,
                                   seed=234,
                                   parallelism=128)
ElasticNet_Model=ElasticNet_CrossVal.fit(spark_power_data)

26/04/27 22:37:38 WARN BlockManager: Block rdd_59_0 already exists on this machine; not re-adding it
26/04/27 22:37:38 WARN BlockManager: Block rdd_59_0 already exists on this machine; not re-adding it
26/04/27 22:37:38 WARN BlockManager: Block rdd_59_0 already exists on this machine; not re-adding it
26/04/27 22:37:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/27 22:37:43 WARN Instrumentation: [35ffb454] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 22:37:43 WARN Instrumentation: [e3e27021] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 22:37:43 WARN Instrumentation: [35b79f61] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 22:37:43 WARN Instrumentation: [cfb96fa5] regParam is zero, which might cause numerical instability and overfitting.
26/0

In [12]:
ElasticNet_RMSEs=[]
for _ in range(len(EN_Param_Grid)):
    ElasticNet_RMSEs.append([ElasticNet_Model.avgMetrics[_],EN_Param_Grid[_].values()])

In [13]:
ElasticNet_RMSEs

[[2147.7095564430283, dict_values([0.0, 0.0])],
 [2147.7095564430283, dict_values([0.0, 0.05])],
 [2147.7095564430283, dict_values([0.0, 0.1])],
 [2147.7095564430283, dict_values([0.0, 0.25])],
 [2147.7095564430283, dict_values([0.0, 0.5])],
 [2147.7095564430283, dict_values([0.0, 0.75])],
 [2147.7095564430283, dict_values([0.0, 0.9])],
 [2147.7095564430283, dict_values([0.0, 0.95])],
 [2147.7095564430283, dict_values([0.0, 0.98])],
 [2147.7095564430283, dict_values([0.0, 0.99])],
 [2147.7095564430283, dict_values([0.0, 1.0])],
 [2147.709539089853, dict_values([0.05, 0.0])],
 [2147.708880531484, dict_values([0.05, 0.05])],
 [2147.707684374583, dict_values([0.05, 0.1])],
 [2147.7088562253643, dict_values([0.05, 0.25])],
 [2147.709717669582, dict_values([0.05, 0.5])],
 [2147.7094348063456, dict_values([0.05, 0.75])],
 [2147.7097292283847, dict_values([0.05, 0.9])],
 [2147.7097297877467, dict_values([0.05, 0.95])],
 [2147.7097373897805, dict_values([0.05, 0.98])],
 [2147.709739948686, dic

Upon close inspection of each RMSE associated with each pair of hyperparameters, we can see that the lowest RMSE goes to the one with the `regParam` of 0.25 and the `ElasticNetParam` of 0.05, indicating fairly minor regularization required for this model. The CV error (i.e. the average RMSE) for this model is shown below.

In [17]:
Best_EN_Params=ParamGridBuilder().addGrid(MLR.regParam,[0.5]).addGrid(MLR.elasticNetParam,[0.05]).build()
Best_ElasticNet_CV=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=Best_EN_Params,
                                   evaluator=RegressionEvaluator(labelCol='label',metricName='rmse'),
                                   numFolds=5)
Best_ElasticNet_Model=Best_ElasticNet_CV.fit(spark_power_data)

In [18]:
ElasticNet_CV_Error=np.mean(ElasticNet_Model.avgMetrics)
ElasticNet_Training_RMSE=Best_ElasticNet_Model.avgMetrics
print(ElasticNet_CV_Error)
print(ElasticNet_Training_RMSE[0])

2147.7235346958432
2147.9360127812915


In [19]:
Best_ElasticNet_Model_Preds=Best_ElasticNet_Model.transform(spark_power_data)
Best_ElasticNet_Model_Preds_Resids=Best_ElasticNet_Model_Preds.withColumn("residuals",col("label")-col("prediction"))
Best_ElasticNet_Model_Preds_Resids.select("label","prediction","residuals").show()

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386|20880.930409918135|-639.9665499181356|
|20131.08434| 18659.20207062018| 1471.882269379821|
|19668.43373| 18203.68706526658|1464.7466647334222|
|18899.27711|17589.615584546333|1309.6615254536664|
|18442.40964|16996.222013493007| 1446.187626506995|
|18130.12048|16516.604010769734|1613.5164692302678|
|17945.06024|16092.172203363683|1852.8880366363155|
|17459.27711|15721.607521707592|1737.6695882924068|
|17025.54217| 15269.95627697898|1755.5858930210197|
|16794.21687|14937.243003824544|1856.9738661754564|
|16638.07229| 14651.40993450192|1986.6623554980797|
|16395.18072|14413.923740901228|1981.2569790987727|
|16117.59036|14081.753300423043|2035.8370595769575|
| 15822.6506|13623.757545348893|2198.8930546511074|
|15672.28916|13449.285811191512| 2223.003348808488|
|15597.10843|13301.292361297285|2295.8160687027157|
|15510.36145

In [55]:
#The overall idea of Best_ElasticNet_Model_Preds_Resids will be saved as a dataframe for later use. Namely

pyspark.ml.tuning.CrossValidatorModel

In [54]:
np.mean(ElasticNet_Model.avgMetrics)

2147.9477178282195